# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = ['0xa01ea43d416a4582a022fb75319ade76645699d8',
 '0x702226b14b65b1c8d05ff51759e095f09932f64e',
 '0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf',
 '0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca',
 '0x522f77e63f087115ee51c7cfda1d34c0b46e1415',
 '0x450eb6572cbdbe5257821cbacd7c64e4ea0380ae',
 '0xcf0aca0d7a395202aec661c3666be9cc098e320a',
 '0x865ab91c322793309506adca4f8f245e0023901b',
 '0xa772656bf37c8a62e4e91ac83beb6a7a3229ae85',
 '0x33c9101c64b9ba0ffcc200408afa2e91891f5124',
 '0x26083bbc3665c6d9a58d947bdca93fe528a834c1',
 '0xd77d68c7b502f2e19885c2fe8a7a9198c86ab01c',
 '0xaf3c7d23f497ea9aa23c522a25fa291274b4f5ff',
 '0xc13f2a97dd2a68061c1b382568b7bd711a08c93b',
 '0xf13bc7526456c46b62dac7ee210a957adbe89ef9',
 '0x3f2c9b3d2d7a09143f810940b4da51ac90b2f0cf',
 '0xd6f44883f664d7dc963d8b89c5a0689fdd330566',
 '0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007',
 '0x4fb060ef365ad6a76d8da1387ec6a7372d4b3182',
 '0xd6e26297e4330f7e444f4c7e3cb5ca1a3a92c751',
 '0x79df9bd7244130221562c5371e99f8a0f7f6277e',
 '0x2687cb380661a0041bf4c4cf3945a034b79e1363',
 '0x638767b84130fe027cb773c204f89a9e8f0d9d9b',
 '0x924459f9dbdac960edfc02ac45e0db92df2f4b6e',
 '0xa924ab525445bfaee728f3c648b4fa87157cb1b9',
 '0x47c6e09427e5581445323964afa8432803e82fe4',
 '0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62',
 '0xbff9efe3a976c115e3e639b4c6b9c7168479009d',
 '0x9d5f391fc63c8a887a8cc3a10fa4b065637084bf',
 '0x1d7f4976c7759e280f1bad4a4f8e8eb191f9f6ed',
 '0xabde435151e5cf858071cec380303acb610fd41f',
 '0x8b0244a0ded9943470a6bdad9050dce24bb1f3d9',
 '0x236599e3745dbea9d285d7ac967846f476d8aaf4',
 '0x3335de45972fd32e5d8a3b689e772147f2693656',
 '0x54fde20fa60e932fc480071e05695b42caa94ac3',
 '0xd7951f1ff7880761dcb3d186630d86047dbf6fcd',
 '0x37bdf9681cecfebe1f5709d2aed823e0063c2a26',
 '0x3532fc27e44816c2dfd9e6f53a425ee67b0e3711',
 '0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35',
 '0xf201a19b43471261a3c1ba9247335d55270e527e',
 '0x3458ae6606828ed4d9666573e1585d9419efb5a7',
 '0xab85b40a0d684026153d45a71039de4eeed081e6',
 '0x0c4f5b1295f39cf505679209a22adbafe61c0f33',
 '0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80',
 '0x6fbbd19e716f7454bdf483c41e6fa276a9e139da']

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 2, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 3, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 100.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

In [2]:
banned_wallets = set()
WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [3]:
import pandas as pd

if not WATCHED_WALLETS:
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
    wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Using 45 manually specified wallets.


## Discover shards and derive test period

In [4]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-02-15
Backtest period : 2026-02-16  →  2026-03-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [5]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    positions: dict[tuple[str, str, str], float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(order_id: int, trig, order_px: float, qty_ordered: float, trig_tw: bool) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)
                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] = max(positions[order["pos_key"]] - fill_qty, 0.0)

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(order_id, trig, order_px, qty_ordered, trig_tw)
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [6]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    :  52,991
Fill rows         : 115,764
Orders with fills :  43,117
Order fill rate   : 81.4%


## Summary statistics

In [7]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    :  52,991
  Orders with fills   :  43,117  (81.4%)
  Orders multi-fill   :  22,758  (52.8% of filled)
  Avg fill %          :    95.0%
  Total qty filled    : 3,613,981.05 tokens
  Total notional      : 1,652,342.14 USDC
  Net PnL (USDC)      : -16,320.27
  Net ROI on notional :    -0.99%
  Win rate (by order) :    50.73%
  Avg exec price      :   0.4905

  Fill source breakdown:
    same_token          : 78,290  (67.6%)
    opposite_token      : 37,474  (32.4%)

=== Watched-wallet cohort ===
  Total trades        :  52,991
  Total trade PnL     :  92,535.38 USDC


## Event ledger preview

In [8]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,exec_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,wallet_trade_pnl
0,trigger,1,0x702226b14b65b1c8d05ff51759e095f09932f64e,0xe9c127a8c35f045d37b5344b0a36711084fa20c2fc16...,No,BUY,2026-02-16 00:00:21+00:00,0xccf1f853f14a4bf3ce75b17f9d300d64eeb661e5653d...,0.963000,0.963000,NaN,6.000000,0.000000,NaN,None,True,NaN,0.222000
1,trigger,2,0x702226b14b65b1c8d05ff51759e095f09932f64e,0xbf79498f6fe674c3959fce48c3c28e5e18b2b205e99f...,No,BUY,2026-02-16 00:06:53+00:00,0x15b907c3d24a2627b71ca0c376e895d64e0bdcad4e05...,0.510000,0.510000,NaN,20.408162,0.000000,NaN,None,True,NaN,9.999999
3,fill,3,0x702226b14b65b1c8d05ff51759e095f09932f64e,0x7a310d9af29782a53ccb16cead33ff3140a696c2bc40...,Yes,BUY,2026-02-16 00:14:35+00:00,0xb8accfc1699cd57fc839eba7fd02350420eff972aa27...,0.080000,NaN,0.100000,16.000000,NaN,2.173912,same_token,False,-0.217609,NaN
2,trigger,3,0x702226b14b65b1c8d05ff51759e095f09932f64e,0x7a310d9af29782a53ccb16cead33ff3140a696c2bc40...,Yes,BUY,2026-02-16 00:13:01+00:00,0x1d5ab668e1a1bd6bf6b660f4ac1da88a1757daabcdaa...,0.100000,0.100000,NaN,16.000000,2.173912,NaN,None,False,NaN,-1.600000
4,trigger,4,0x702226b14b65b1c8d05ff51759e095f09932f64e,0xaf1028426fa2b9c594a06239665c75b9a4af08c3c875...,No,BUY,2026-02-16 00:18:05+00:00,0x8cabd83122bf2495611fa913ae2283e1240535ca88a2...,0.490000,0.490000,NaN,10.000000,0.000000,NaN,None,False,NaN,-4.900000
6,fill,5,0x702226b14b65b1c8d05ff51759e095f09932f64e,0x2850353da31ae0f2ba9c2534922aaad13631850dead6...,No,BUY,2026-02-16 00:21:01+00:00,0x1e6d23e671fb4589fccaaa2b52ab454a3cdad8a09332...,0.380000,NaN,0.383529,17.000000,NaN,15.510000,opposite_token,True,9.555510,NaN
7,fill,5,0x702226b14b65b1c8d05ff51759e095f09932f64e,0x2850353da31ae0f2ba9c2534922aaad13631850dead6...,No,BUY,2026-02-16 00:21:01+00:00,0x1e6d23e671fb4589fccaaa2b52ab454a3cdad8a09332...,0.380000,NaN,0.383529,17.000000,NaN,1.490000,opposite_token,True,0.917970,NaN
5,trigger,5,0x702226b14b65b1c8d05ff51759e095f09932f64e,0x2850353da31ae0f2ba9c2534922aaad13631850dead6...,No,BUY,2026-02-16 00:19:57+00:00,0xa81f673548b18764f1aa53a9fdbffa514bfc3e291ce4...,0.383529,0.383529,NaN,17.000000,17.000000,NaN,None,True,NaN,10.480000


In [9]:
trade_df = pd.read_parquet(RAW_TRADES_DIR / "3.parquet")
len(trade_df)

4211083

In [10]:
trade_df[trade_df["tx_hash"] == "0xa351e21d93a55a53c795aefeb1d562f616b17610d10e008c8dfb074caa17504b"]

,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,token_winner,wallet,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
2263478,0xa351e21d93a55a53c795aefeb1d562f616b17610d10e...,1892,83040166,1771200355,2026-02-16,0x32cad3ff44a2ab9a9f1b6eff9d405fb3f265959389bf...,5176937146910646254310958050143677964494788696...,Over,False,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,...,1,3622.560511,True,False,"[Sports, Soccer, Games, Liga MX]",BUY,Under,1444043070393945705294311409642149971788542593...,0.401,82.966900
2263479,0xa351e21d93a55a53c795aefeb1d562f616b17610d10e...,1892,83040166,1771200355,2026-02-16,0x32cad3ff44a2ab9a9f1b6eff9d405fb3f265959389bf...,5176937146910646254310958050143677964494788696...,Over,False,0xa1783e4d22e10afee1b9828c1d1e73f33e2211a2,...,1,206.900000,True,False,"[Sports, Soccer, Games, Liga MX]",SELL,Under,1444043070393945705294311409642149971788542593...,0.401,82.966900
2263480,0xa351e21d93a55a53c795aefeb1d562f616b17610d10e...,1900,83040166,1771200355,2026-02-16,0x32cad3ff44a2ab9a9f1b6eff9d405fb3f265959389bf...,1444043070393945705294311409642149971788542593...,Under,True,0x77bcf759a5bbbf604a52d839d5f68ead4c78b91b,...,1,1339.557437,False,False,"[Sports, Soccer, Games, Liga MX]",BUY,Under,1444043070393945705294311409642149971788542593...,0.400,17.377933
2263481,0xa351e21d93a55a53c795aefeb1d562f616b17610d10e...,1900,83040166,1771200355,2026-02-16,0x32cad3ff44a2ab9a9f1b6eff9d405fb3f265959389bf...,5176937146910646254310958050143677964494788696...,Over,False,0xa1783e4d22e10afee1b9828c1d1e73f33e2211a2,...,1,250.344832,True,False,"[Sports, Soccer, Games, Liga MX]",SELL,Under,1444043070393945705294311409642149971788542593...,0.400,17.377933


## Build PnL time series

In [11]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 615
Copy strategy hourly buckets : 605


## Cumulative PnL chart

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [13]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,40319,32654,88427,81.0
SELL,12672,10463,27337,82.6


In [14]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,24319
1,BUY,same_token,64108
2,SELL,opposite_token,13155
3,SELL,same_token,14182


In [15]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0xabde435151e5cf858071cec380303acb610fd41f,1497,4426,6168.739850,11260.409012,6168.739850,1452,0.969940
0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35,3849,6497,4072.901806,-9297.936083,4072.901806,3059,0.794752
0x37bdf9681cecfebe1f5709d2aed823e0063c2a26,256,560,2668.185988,6948.277356,2668.185988,256,1.000000
0x702226b14b65b1c8d05ff51759e095f09932f64e,4441,3982,2422.922010,-2530.795836,2422.922010,2253,0.507318
0xcf0aca0d7a395202aec661c3666be9cc098e320a,1889,4235,1064.586832,9685.177744,1064.586832,1639,0.867655


In [16]:
len(df)

168755

In [17]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [18]:
merged_df[merged_df['diff_dt'].notnull()]['diff_dt'].mean().total_seconds()

33.102094

In [19]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0xf201a19b43471261a3c1ba9247335d55270e527e',
 '0xd6f44883f664d7dc963d8b89c5a0689fdd330566',
 '0x236599e3745dbea9d285d7ac967846f476d8aaf4',
 '0xbff9efe3a976c115e3e639b4c6b9c7168479009d',
 '0xaf3c7d23f497ea9aa23c522a25fa291274b4f5ff',
 '0x0c4f5b1295f39cf505679209a22adbafe61c0f33',
 '0x8b0244a0ded9943470a6bdad9050dce24bb1f3d9',
 '0x3f2c9b3d2d7a09143f810940b4da51ac90b2f0cf',
 '0x47c6e09427e5581445323964afa8432803e82fe4',
 '0x6fbbd19e716f7454bdf483c41e6fa276a9e139da',
 '0xa924ab525445bfaee728f3c648b4fa87157cb1b9',
 '0x79df9bd7244130221562c5371e99f8a0f7f6277e',
 '0x33c9101c64b9ba0ffcc200408afa2e91891f5124',
 '0xc13f2a97dd2a68061c1b382568b7bd711a08c93b',
 '0x3458ae6606828ed4d9666573e1585d9419efb5a7',
 '0x924459f9dbdac960edfc02ac45e0db92df2f4b6e',
 '0x26083bbc3665c6d9a58d947bdca93fe528a834c1',
 '0xa772656bf37c8a62e4e91ac83beb6a7a3229ae85',
 '0x522f77e63f087115ee51c7cfda1d34c0b46e1415',
 '0x9d5f391fc63c8a887a8cc3a10fa4b065637084bf',
 '0x450eb6572cbdbe5257821cbacd7c64e4ea0380ae',
 '0x865ab91c3

In [20]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0xabde435151e5cf858071cec380303acb610fd41f,1452,4426,6168.739850,1.038490e+05,73.1
1,0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35,3059,6497,4072.901806,2.045955e+05,58.2
2,0x37bdf9681cecfebe1f5709d2aed823e0063c2a26,256,560,2668.185988,3.667209e+04,56.6
3,0x702226b14b65b1c8d05ff51759e095f09932f64e,2253,3982,2422.922010,7.626720e+04,50.4
4,0xcf0aca0d7a395202aec661c3666be9cc098e320a,1639,4235,1064.586832,1.612199e+05,44.8
5,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,318,750,1020.744783,2.488394e+04,42.8
6,0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca,695,1745,846.143306,2.943989e+04,58.3
7,0x2687cb380661a0041bf4c4cf3945a034b79e1363,1995,4854,727.530274,1.611389e+05,81.6
8,0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80,5318,13685,507.905636,3.005636e+05,50.8
9,0x3335de45972fd32e5d8a3b689e772147f2693656,111,478,397.731634,1.526993e+04,45.9


In [35]:
wallet_pnl_df.head(10)['wallet'].tolist()

['0xabde435151e5cf858071cec380303acb610fd41f',
 '0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35',
 '0x37bdf9681cecfebe1f5709d2aed823e0063c2a26',
 '0x702226b14b65b1c8d05ff51759e095f09932f64e',
 '0xcf0aca0d7a395202aec661c3666be9cc098e320a',
 '0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62',
 '0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca',
 '0x2687cb380661a0041bf4c4cf3945a034b79e1363',
 '0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80',
 '0x3335de45972fd32e5d8a3b689e772147f2693656']

In [21]:
event_ledger[event_ledger["wallet"] == "0xd6f44883f664d7dc963d8b89c5a0689fdd330566"]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard
13616,trigger,4892,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x69c1be8e2e61776d54dda52a43de0b1fc73bb27b153b...,Ball State Cardinals,BUY,2026-02-20 23:45:32+00:00,0x42e052941a45658e843727655e4afef36a30cd890bf5...,0.129736,0.129736,300.000000,300.000000,NaN,None,False,NaN,NaN,-38.9209,6
13617,fill,4892,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x69c1be8e2e61776d54dda52a43de0b1fc73bb27b153b...,Ball State Cardinals,BUY,2026-02-20 23:45:44+00:00,0x89084348ceb5fe7a3ea47168dbaf7f949446950a4a6e...,0.110000,NaN,300.000000,NaN,11.99,opposite_token,False,0.129736,-1.557094,NaN,6
13618,fill,4892,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x69c1be8e2e61776d54dda52a43de0b1fc73bb27b153b...,Ball State Cardinals,BUY,2026-02-20 23:45:56+00:00,0xae28b56cb0bddae132c01799e93231e53269edd5e08c...,0.100000,NaN,300.000000,NaN,10.00,opposite_token,False,0.129736,-1.298661,NaN,6
13619,fill,4892,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x69c1be8e2e61776d54dda52a43de0b1fc73bb27b153b...,Ball State Cardinals,BUY,2026-02-20 23:45:56+00:00,0xae28b56cb0bddae132c01799e93231e53269edd5e08c...,0.100000,NaN,300.000000,NaN,249.00,opposite_token,False,0.129736,-32.336651,NaN,6
13620,trigger,4893,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x69c1be8e2e61776d54dda52a43de0b1fc73bb27b153b...,Ball State Cardinals,SELL,2026-02-20 23:45:56+00:00,0xae28b56cb0bddae132c01799e93231e53269edd5e08c...,0.100000,0.100000,111.111111,111.111111,NaN,None,False,NaN,NaN,25.9000,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167559,fill,52610,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x0abf2af0902a3d6c883bae52b1a50ed4aa6b937d4e9b...,Canucks,SELL,2026-03-13 04:36:01+00:00,0xb45477c2fce47b758e562a6378544ce8f05363787881...,0.500000,NaN,45.000000,NaN,3.78,same_token,True,0.500000,-1.891890,NaN,0
167560,fill,52610,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x0abf2af0902a3d6c883bae52b1a50ed4aa6b937d4e9b...,Canucks,SELL,2026-03-13 04:36:01+00:00,0x8dc60bb07e8c877d0d0afc8573c9a91190bce9ba372f...,0.500000,NaN,45.000000,NaN,3.88,same_token,True,0.500000,-1.941940,NaN,0
167561,fill,52610,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x0abf2af0902a3d6c883bae52b1a50ed4aa6b937d4e9b...,Canucks,SELL,2026-03-13 04:36:01+00:00,0x42166b396ddbdb45edb15891e5e6f30b3aa5bff75cab...,0.500000,NaN,45.000000,NaN,2.20,same_token,True,0.500000,-1.101100,NaN,0
167562,fill,52610,0xd6f44883f664d7dc963d8b89c5a0689fdd330566,0x0abf2af0902a3d6c883bae52b1a50ed4aa6b937d4e9b...,Canucks,SELL,2026-03-13 04:36:03+00:00,0x879f7667cb8210ee3fb56c55561ad5caa9d6a8e33241...,0.500000,NaN,45.000000,NaN,0.66,opposite_token,True,0.500000,-0.330330,NaN,0


In [22]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 9,874 (18.6% of all triggers)
Partially filled   : 3,498 (8.1% of filled orders)
Fully filled       : 39,619

Unfilled breakdown by side:


,side,count
0,BUY,7665
1,SELL,2209


In [23]:
triggers[triggers['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard
53,trigger,34,0x702226b14b65b1c8d05ff51759e095f09932f64e,0xfa1a019d6e3ee284805ffdbee54b8a39a220c32a9bb1...,No,BUY,2026-02-16 01:35:29+00:00,0x24349b840a63013f6025b7de2139aa1e761f8cb6176c...,0.57,0.57,6.0,6.0,NaN,None,False,NaN,NaN,-3.42,f


In [24]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard
54,fill,34,0x702226b14b65b1c8d05ff51759e095f09932f64e,0xfa1a019d6e3ee284805ffdbee54b8a39a220c32a9bb1962bef81436ffbd87426,No,BUY,2026-02-16 01:35:53+00:00,0x7b896d916f365e1b1f1776c784e8c0c5b88334271cf56bb649550192b2589fc2,0.55,NaN,6.0,NaN,6.0,same_token,False,0.57,-3.42342,NaN,f


In [25]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False).head(20)

wallet
0xabde435151e5cf858071cec380303acb610fd41f    6168.739850
0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35    4072.901806
0x37bdf9681cecfebe1f5709d2aed823e0063c2a26    2668.185988
0x702226b14b65b1c8d05ff51759e095f09932f64e    2422.922010
0xcf0aca0d7a395202aec661c3666be9cc098e320a    1064.586832
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62    1020.744783
0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca     846.143306
0x2687cb380661a0041bf4c4cf3945a034b79e1363     727.530274
0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80     507.905636
0x3335de45972fd32e5d8a3b689e772147f2693656     397.731634
0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf     290.580686
0x54fde20fa60e932fc480071e05695b42caa94ac3     249.357109
0xd77d68c7b502f2e19885c2fe8a7a9198c86ab01c     228.868166
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007     221.847522
0x1d7f4976c7759e280f1bad4a4f8e8eb191f9f6ed     135.344938
0xf13bc7526456c46b62dac7ee210a957adbe89ef9      74.852220
0xab85b40a0d684026153d45a71039de4eeed081e6      -2.207128
0x63876

In [26]:
bad_wallets = ['0xdf0850b029c97de282c491e91ce500384782cf97',
 '0xbb50fa724ac755197246d950008d6a0564ee7bfc',
 '0xf0b0ef1d6320c6be896b4c9c54dd74407e7f8cab',
 '0x0cbb0a19fe967a735f6659d2a39338161cea59fc',
 '0xad5353afe30c2da57709e2704ef3ccdcf67eef24',
 '0x87650b9f63563f7c456d9bbcceee5f9faf06ed81',
 '0x75d1b199a96801338e67a6efaa35c0028ea87a5f',
 '0x0a6d26d31b28fd5a84c301f8b27296612f3b1d0a',
 '0xd6f44883f664d7dc963d8b89c5a0689fdd330566',
 '0x77bcf759a5bbbf604a52d839d5f68ead4c78b91b']

fills[~fills["wallet"].isin(bad_wallets)]['fill_pnl_usdc'].sum()

np.float64(-4841.911917548954)

In [27]:
filled_orders = fills["order_id"].unique()

In [28]:
pd.set_option("display.max_rows", None)
event_ledger[event_ledger["order_id"].isin(filled_orders)].sort_values('dt')[['row_type', 'order_id', 'dt', 'side', 'price', 'wallet_trade_pnl', 'fill_pnl_usdc', 'token_winner', 'condition_id', 'outcome', 'wallet']].head(5)

,row_type,order_id,dt,side,price,wallet_trade_pnl,fill_pnl_usdc,token_winner,condition_id,outcome,wallet
2,trigger,3,2026-02-16 00:13:01+00:00,BUY,0.100000,-1.60,NaN,False,0x7a310d9af29782a53ccb16cead33ff3140a696c2bc40a79107abd2dde41ecf8f,Yes,0x702226b14b65b1c8d05ff51759e095f09932f64e
3,fill,3,2026-02-16 00:14:35+00:00,BUY,0.080000,NaN,-0.217609,False,0x7a310d9af29782a53ccb16cead33ff3140a696c2bc40a79107abd2dde41ecf8f,Yes,0x702226b14b65b1c8d05ff51759e095f09932f64e
5,trigger,5,2026-02-16 00:19:57+00:00,BUY,0.383529,10.48,NaN,True,0x2850353da31ae0f2ba9c2534922aaad13631850dead69264af71a345d1d1ae78,No,0x702226b14b65b1c8d05ff51759e095f09932f64e
6,fill,5,2026-02-16 00:21:01+00:00,BUY,0.380000,NaN,9.555510,True,0x2850353da31ae0f2ba9c2534922aaad13631850dead69264af71a345d1d1ae78,No,0x702226b14b65b1c8d05ff51759e095f09932f64e
7,fill,5,2026-02-16 00:21:01+00:00,BUY,0.380000,NaN,0.917970,True,0x2850353da31ae0f2ba9c2534922aaad13631850dead69264af71a345d1d1ae78,No,0x702226b14b65b1c8d05ff51759e095f09932f64e


In [29]:
trigger_wallets = triggers[["order_id", "wallet"]].drop_duplicates().set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x0c4f5b1295f39cf505679209a22adbafe61c0f33,0xe11e8536e0074711e4ad830e5b579653e6a7a291b3b21c45390bf944355af4d0,1500.779238,45


In [30]:
(
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
    )
    .groupby("wallet")
    .agg(
        total_pnl=("pnl", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("total_pnl", ascending=False)
    .head(100)
)

,total_pnl,unique_conditions,total_orders_filled
wallet,,,
0xabde435151e5cf858071cec380303acb610fd41f,6168.739850,154,1452
0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35,4072.901806,193,3059
0x37bdf9681cecfebe1f5709d2aed823e0063c2a26,2668.185988,65,256
0x702226b14b65b1c8d05ff51759e095f09932f64e,2422.922010,165,2253
0xcf0aca0d7a395202aec661c3666be9cc098e320a,1064.586832,223,1639
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,1020.744783,31,318
0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca,846.143306,321,695
0x2687cb380661a0041bf4c4cf3945a034b79e1363,727.530274,227,1995
0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80,507.905636,1774,5318


In [31]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [32]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0x37bdf9681cecfebe1f5709d2aed823e0063c2a26,65,53.746154
0xf201a19b43471261a3c1ba9247335d55270e527e,107,45.190893
0xabde435151e5cf858071cec380303acb610fd41f,154,34.714299
0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35,193,27.261501
0x3458ae6606828ed4d9666573e1585d9419efb5a7,66,10.961056
0x2687cb380661a0041bf4c4cf3945a034b79e1363,227,9.608229
0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,159,8.171818
0x522f77e63f087115ee51c7cfda1d34c0b46e1415,78,5.694991
0x865ab91c322793309506adca4f8f245e0023901b,64,4.970770
